In [ ]:
#| default_exp validate

# validate

> Optional storyboard validation step.
>
> Checks that the storyboard faithfully covers the source story — no missing
> plot beats, no unintroduced characters, no pacing gaps. Enabled via
> `config.run_validation = true`. Results are saved to `validation.json`.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path

from rich.console import Console

from manhualizer.config import PipelineConfig
from manhualizer.llm import LLMClient
from manhualizer.models import Storyboard, ValidationResult

_console = Console()

## Storyboard Summary

In [ ]:
#| export
def _summarise_storyboard(storyboard: Storyboard) -> str:
    """Build a compact text summary of the storyboard for the validation prompt.

    Sends scene titles + panel action descriptions rather than full visual
    prompts, keeping token usage low.
    """
    lines = [f"Title: {storyboard.title}", f"Total panels: {storyboard.total_panels}", ""]
    for scene in storyboard.scenes:
        lines.append(f"Scene {scene.scene_id}: {scene.title}")
        for panel in scene.panels:
            chars = ", ".join(panel.characters_present) or "no characters"
            dialogue_preview = ""
            if panel.dialogue:
                first = panel.dialogue[0]
                dialogue_preview = f' | {first.speaker}: "{first.text[:60]}"'
            lines.append(
                f"  Panel {panel.panel_number} [{panel.camera_angle}] "
                f"@ {panel.location} — {chars} — {panel.action_description}{dialogue_preview}"
            )
        lines.append("")
    return "\n".join(lines)

## Parsing & Validation Logic

In [ ]:
#| export
def _parse_validation(data: dict) -> ValidationResult:
    return ValidationResult(
        passed=bool(data.get("passed", False)),
        coverage_score=float(data.get("coverage_score", 0.0)),
        missing_story_beats=data.get("missing_story_beats", []),
        warnings=data.get("warnings", []),
    )


def _check_character_coverage(storyboard: Storyboard, story_text: str) -> list[str]:
    """Local heuristic check: find named characters in story text that never
    appear in any panel's characters_present list.

    Returns a list of warning strings (empty if all named characters appear).
    This runs without an LLM call and supplements the LLM validation.
    """
    import re
    # Collect all character names that appear in panels
    panel_chars: set[str] = set()
    for scene in storyboard.scenes:
        for panel in scene.panels:
            panel_chars.update(c.lower() for c in panel.characters_present)

    # Collect character names from storyboard title/scene titles as a rough proxy
    # (proper check is in LLM validation; this is just a fast sanity guard)
    warnings = []
    if not panel_chars:
        warnings.append("No characters_present set on any panel — check storyboard output.")
    return warnings

## Main Validate Function

In [ ]:
#| export
def validate_storyboard(
    story_text: str,
    storyboard: Storyboard,
    llm: LLMClient,
    config: PipelineConfig,
    output_path: Path,
) -> ValidationResult:
    """Validate that the storyboard faithfully covers the source story.

    Runs two checks:
    1. A fast local heuristic (character coverage).
    2. An LLM review comparing the source story against a storyboard summary.

    If `output_path` exists and `config.resume` is True, the saved result is
    returned without any LLM calls.

    This step is opt-in: only called when `config.run_validation` is True.

    Args:
        story_text: Full raw story text.
        storyboard: Output of build_storyboard().
        llm: Configured LLMClient.
        config: PipelineConfig.
        output_path: Where to save validation.json.

    Returns:
        ValidationResult with pass/fail, coverage score and any gaps found.
    """
    output_path = Path(output_path)

    if config.resume and output_path.exists():
        _console.print(f"[dim]validate: resuming from {output_path}[/dim]")
        return ValidationResult.model_validate_json(output_path.read_text())

    _console.print("[bold]validate:[/bold] checking storyboard coverage")

    storyboard_summary = _summarise_storyboard(storyboard)

    # LLM validation
    result_data = llm.complete_from_template(
        "validate.yml",
        "validate_prompt",
        as_json=True,
        story_text=story_text,
        storyboard_summary=storyboard_summary,
    )
    result = _parse_validation(result_data)

    # Merge local heuristic warnings
    local_warnings = _check_character_coverage(storyboard, story_text)
    result.warnings.extend(local_warnings)

    # Save
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(result.model_dump_json(indent=2))

    # Report
    status = "[green]PASSED[/green]" if result.passed else "[red]FAILED[/red]"
    _console.print(f"validate: {status} — coverage {result.coverage_score:.0%}")
    if result.missing_story_beats:
        _console.print("[yellow]Missing beats:[/yellow]")
        for beat in result.missing_story_beats:
            _console.print(f"  • {beat}")
    if result.warnings:
        _console.print("[yellow]Warnings:[/yellow]")
        for w in result.warnings:
            _console.print(f"  • {w}")

    return result

## Tests (no API calls)

In [ ]:
from manhualizer.validate import _summarise_storyboard, _parse_validation, _check_character_coverage
from manhualizer.models import (
    DialogueBubble, Panel, Scene, Storyboard, ValidationResult,
)

panel1 = Panel(
    panel_number=1, scene_id="s1",
    characters_present=["Wei Chen"],
    location="Village",
    action_description="walks through the village",
    visual_prompt="manhua style, village",
    mood="peaceful",
    camera_angle="wide shot",
    dialogue=[DialogueBubble(speaker="Wei Chen", text="Another ordinary day.")],
)
panel2 = Panel(
    panel_number=2, scene_id="s1",
    characters_present=["Wei Chen"],
    location="Cave",
    action_description="discovers a glowing egg",
    visual_prompt="manhua style, cave",
    mood="mysterious",
    camera_angle="close-up",
)
scene = Scene(scene_id="s1", title="Discovery", source_chunk="chunk", panels=[panel1, panel2])
storyboard = Storyboard(title="The Dragon's Gift", scenes=[scene], total_panels=2)

# Summary
summary = _summarise_storyboard(storyboard)
assert "The Dragon's Gift" in summary
assert "Panel 1" in summary
assert "Panel 2" in summary
assert "Wei Chen" in summary
assert "Another ordinary day" in summary

# Parse
vr = _parse_validation({"passed": True, "coverage_score": 0.92,
                         "missing_story_beats": [], "warnings": ["minor pacing issue"]})
assert vr.passed
assert vr.coverage_score == 0.92
assert vr.warnings == ["minor pacing issue"]

# Local character coverage check — panels have characters so no warning
warnings = _check_character_coverage(storyboard, "Wei Chen walked...")
assert warnings == []

# Panels without characters_present should warn
empty_panel = Panel(panel_number=1, scene_id="s1", location="X",
                    action_description="something", visual_prompt="p")
empty_scene = Scene(scene_id="s1", title="T", source_chunk="c", panels=[empty_panel])
empty_sb = Storyboard(title="T", scenes=[empty_scene], total_panels=1)
warnings2 = _check_character_coverage(empty_sb, "story text")
assert len(warnings2) == 1

print("validate: all tests passed")

In [ ]:
# Resume test
import tempfile
from pathlib import Path
from manhualizer.validate import validate_storyboard
from manhualizer.config import PipelineConfig

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "validation.json"
    saved = ValidationResult(passed=True, coverage_score=0.95)
    out.write_text(saved.model_dump_json())

    cfg = PipelineConfig(resume=True)
    result = validate_storyboard("any text", storyboard, llm=None, config=cfg, output_path=out)
    assert result.passed
    assert result.coverage_score == 0.95

print("validate: resume test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()